# Phase 6 DAEAC Hybrid MK-MMD + MCC - Strategy A Focal Alpha

Kaggle driver for `phase6_daeac_hybrid_mkmmd_mcc_focal_alpha.yaml`.

Strategy A uses alpha-balanced focal loss for the supervised source classification term: `gamma=2.35`, `alpha=[1.6, 1.8, 0.8, 1.0]` for classes `N/S/V/F`.

## What To Attach In Kaggle

Attach these Kaggle inputs before running:

1. **Preprocessed DAEAC data**: a dataset folder containing `mitdb_ds1_daeac.npz`, `mitdb_ds2_first5_unlabeled_daeac.npz`, `mitdb_ds2_daeac.npz`, and optionally `incart_all_daeac.npz`, `svdb_all_daeac.npz`. A folder named `daeac/` is preferred, but the notebook also searches all `/kaggle/input/**/*.npz`.
2. **DAEAC base checkpoint**: a dataset containing `daeac_base_best.pt`. The focal experiment starts from this source-trained base checkpoint. If you do not attach it, set `TRAIN_BASE_IF_MISSING = True`, but that can take a long time.
3. **Repository access**: either enable Internet and set `REPO_URL`, or upload this repo as a Kaggle dataset and adjust `REPO`/`ECG` manually.

Recommended Kaggle settings: GPU accelerator, Internet on for clone/install, sufficient disk for `outputs/`.

In [ ]:
from pathlib import Path
import os, shutil

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'  # edit before cloning
BRANCH = 'main'

STRATEGY = 'focal_alpha'
CONFIG = 'configs/phase6_daeac_hybrid_mkmmd_mcc_focal_alpha.yaml'
PREFIX = 'daeac_hybrid_mkmmd_mcc_focal_alpha'
OUTPUT_DIR = Path('outputs/phase6_daeac_hybrid_mkmmd_mcc_focal_alpha')
EVAL_DATASET = 'all'  # requires incart_all_daeac.npz and svdb_all_daeac.npz in Kaggle inputs

TRAIN_BASE_IF_MISSING = False
SMOKE_SAMPLES = 512

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Strategy:', STRATEGY)
print('Repo path:', ECG)

## 1. Clone Or Reuse Repo

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)

assert ECG.exists(), f'Missing repo at {ECG}'
os.chdir(ECG)
print('cwd:', Path.cwd())
!git status --short || true

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Copy DAEAC Data From Kaggle Inputs

In [ ]:
DEST = Path('data/processed/phase6_daeac_paper')
DEST.mkdir(parents=True, exist_ok=True)
required_npz = [
    'mitdb_ds1_daeac.npz',
    'mitdb_ds2_first5_unlabeled_daeac.npz',
    'mitdb_ds2_daeac.npz',
]
optional_npz = ['incart_all_daeac.npz', 'svdb_all_daeac.npz']

candidate_dirs = [p for p in Path('/kaggle/input').glob('**/daeac') if p.is_dir()]
print('Candidate daeac dirs:', candidate_dirs)

copied = set()
search_roots = candidate_dirs if candidate_dirs else [Path('/kaggle/input')]
for filename in required_npz + optional_npz:
    matches = []
    for root in search_roots:
        matches.extend(root.glob(f'**/{filename}'))
    if matches:
        src = matches[0]
        shutil.copy2(src, DEST / filename)
        copied.add(filename)
        print('copied', src, '->', DEST / filename)
    else:
        print('not found:', filename)

missing = [name for name in required_npz if not (DEST / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing required DAEAC npz files: {missing}. Attach the preprocessed DAEAC Kaggle dataset.')

!ls -lh data/processed/phase6_daeac_paper

## 4. Copy Or Train Base Checkpoint

The hybrid MK-MMD+MCC adaptation config expects `outputs/phase6_daeac_paper/checkpoints/daeac_base_best.pt`. Attach this checkpoint if possible. Training it inside this notebook is optional and slow.

In [ ]:
BASE_DEST = Path('outputs/phase6_daeac_paper/checkpoints/daeac_base_best.pt')
BASE_DEST.parent.mkdir(parents=True, exist_ok=True)

if not BASE_DEST.exists():
    matches = list(Path('/kaggle/input').glob('**/daeac_base_best.pt'))
    if matches:
        shutil.copy2(matches[0], BASE_DEST)
        print('copied base checkpoint:', matches[0], '->', BASE_DEST)
    elif TRAIN_BASE_IF_MISSING:
        print('Base checkpoint missing; training source base from config. This may take a long time.')
        !python scripts/phase6_daeac_paper/01_train_base.py --config configs/phase6_daeac_paper.yaml
    else:
        raise FileNotFoundError('Missing daeac_base_best.pt. Attach it as Kaggle input or set TRAIN_BASE_IF_MISSING=True.')

assert BASE_DEST.exists(), f'Missing base checkpoint at {BASE_DEST}'
!ls -lh outputs/phase6_daeac_paper/checkpoints

## 5. Validate Setup

In [ ]:
!python scripts/check_repo.py
!python scripts/phase6_daeac_paper/00_validate_data.py --config {CONFIG}
!python -B -m unittest discover -s tests || true

import yaml
cfg = yaml.safe_load(Path(CONFIG).read_text())
print('source_loss:', cfg['adaptation'].get('source_loss'))
print('focal_gamma:', cfg['adaptation'].get('focal_gamma'))
print('focal_alpha:', cfg['adaptation'].get('focal_alpha'))
print('output_dir:', cfg['paths']['output_dir'])
print('checkpoint_prefix:', cfg['adaptation']['checkpoint_prefix'])

## 6. Smoke Run

Run this before the full experiment. It uses a separate checkpoint prefix ending in `_smoke`.

In [ ]:
!python scripts/phase6_daeac_mcc/02_train_hybrid_mkmmd_mcc.py \
  --config {CONFIG} \
  --epochs 1 \
  --max-source-samples {SMOKE_SAMPLES} \
  --max-target-samples {SMOKE_SAMPLES} \
  --max-val-samples {SMOKE_SAMPLES} \
  --checkpoint-prefix {PREFIX}_smoke

## 7. Full Adaptation Run

In [ ]:
!python scripts/phase6_daeac_mcc/02_train_hybrid_mkmmd_mcc.py --config {CONFIG}

## 8. Evaluate Best And Latest Checkpoints

In [ ]:
best_ckpt = OUTPUT_DIR / 'checkpoints' / f'{PREFIX}_best.pt'
latest_ckpt = OUTPUT_DIR / 'checkpoints' / f'{PREFIX}_latest.pt'
print('best:', best_ckpt, best_ckpt.exists())
print('latest:', latest_ckpt, latest_ckpt.exists())

if best_ckpt.exists():
    !python scripts/phase6_daeac_paper/03_eval.py --config {CONFIG} --checkpoint {best_ckpt} --method-name {PREFIX}_best --dataset {EVAL_DATASET}
if latest_ckpt.exists():
    !python scripts/phase6_daeac_paper/03_eval.py --config {CONFIG} --checkpoint {latest_ckpt} --method-name {PREFIX}_latest --dataset {EVAL_DATASET}

## 9. Zip Outputs For Download

In [ ]:
!zip -r /kaggle/working/phase6_daeac_hybrid_mkmmd_mcc_focal_alpha_outputs.zip outputs/phase6_daeac_hybrid_mkmmd_mcc_focal_alpha outputs/phase6_daeac_paper/checkpoints/daeac_base_best.pt